# Record a run (results)

This notebook demonstrates `results`. It:

1. Seals the inputs, hashing them before the run rather than after.
2. Records what the run produced, and binds a manuscript claim to it.
3. Refuses a second run recorded under an id that already exists.
4. Records that the outcomes were seen.
5. Refuses to call a later run confirmatory, and reports the ordering when forced anyway.

Every cell runs the **published package**.

## Install

In [ ]:
import piplite

await piplite.install(["results-cli==0.2.0"])
print("installed")

## The shell stand-in

`results seal data.csv` becomes `cli('results.cli', 'seal', 'data.csv')`.

In [ ]:
import sys


def cli(module, *args):
    old = sys.argv
    sys.argv = [module.split(".")[0], *args]
    try:
        __import__(module, fromlist=["main"]).main()
    except SystemExit:
        pass
    finally:
        sys.argv = old


print("ready")

## A project with something to seal

In [ ]:
import json
import os
import pathlib

os.makedirs("/tmp/results-demo", exist_ok=True)
os.chdir("/tmp/results-demo")
pathlib.Path("data.csv").write_text("id,arm,outcome\n1,t,0\n2,c,1\n3,t,0\n")
pathlib.Path("analysis.py").write_text("print('pretend this fits a model')\n")

cli("results.cli", "init")

## Seal the inputs, then record what the run produced

Sealing hashes a file *before* the run, so a data file edited afterwards no longer matches
what the analysis was performed on.

In [ ]:
cli("results.cli", "seal", "data.csv", "--role", "data")
cli("results.cli", "seal", "analysis.py", "--role", "script")

pathlib.Path("out.json").write_text(json.dumps({"mortality_90d": 0.213}, indent=2))
cli("results.cli", "run", "out.json", "--run-id", "exp_001", "--note", "primary analysis")

## Bind a manuscript claim to the run that produced it

In [ ]:
cli(
    "results.cli",
    "claim",
    "90-day mortality was 21.3%",
    "--run-id",
    "exp_001",
    "--confirmatory",
    "--location",
    "Table 2",
)

## One run id names one run

Typing the same id twice would let a claim rest on whichever of the two the ledger resolved
first, which defeats the ordering guard below.

In [ ]:
cli("results.cli", "run", "out.json", "--run-id", "exp_001")

---
# The ordering guard

A confirmatory claim asserts an outcome that was fixed *before* anyone looked. Once you
record that the outcomes were seen, a run performed afterwards can no longer back one.

## Record that the outcomes were seen

In [ ]:
cli(
    "results.cli",
    "access",
    "read the mortality table over someone's shoulder",
    "--level",
    "outcomes seen",
)

## A run recorded after that cannot be confirmatory

The run is recorded either way — what is refused is *calling it confirmatory*. Nothing is
deleted and nothing is hidden.

In [ ]:
pathlib.Path("out2.json").write_text(json.dumps({"mortality_90d": 0.198}, indent=2))
cli("results.cli", "run", "out2.json", "--run-id", "exp_002", "--note", "re-run after unblinding")

print("--- claiming it as confirmatory ---")
cli("results.cli", "claim", "90-day mortality was 19.8%", "--run-id", "exp_002", "--confirmatory")

## Recording it anyway is possible, and permanent

`--anyway` does not suppress the finding. The ordering is written into the ledger and
`verify` reports it from then on.

In [ ]:
cli(
    "results.cli",
    "claim",
    "90-day mortality was 19.8%",
    "--run-id",
    "exp_002",
    "--confirmatory",
    "--anyway",
)

---
## The ledger verifies as a chain

Each line carries the hash of the line before it, and a separate anchor records how many
lines there should be. Editing a line breaks the chain; deleting the last one forges no
hash at all, which is why the length is anchored separately.

In [ ]:
cli("results.cli", "verify")